### **D2 / Image Features**

In [1]:
import numpy as np
from src.gmm import GMM
from src.knn import KNN


### **Load Data**

In [2]:
def load_split(split):
    X = np.loadtxt(f"./data/image_features/{split}_data.csv", delimiter=",")
    y = np.loadtxt(f"./data/image_features/{split}_label.csv", delimiter=",").astype(int)
    return X, y


X_train, y_train = load_split("train")
X_val, y_val = load_split("val")
X_test, y_test = load_split("test")


In [3]:
classes = np.unique(y_train)
print("Class order:", classes)
print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)


Class order: [0 1 2 3 4]
Train: (2000, 36)
Validation: (500, 36)
Test: (500, 36)


### **GMM**

In [4]:
models = []
priors = []
for label in classes:
    samples = X_train[y_train == label]
    model = GMM(K=3, reg=1e-3, max_iter=100, tol=1e-4)
    model.fit(samples)
    models.append(model)
    priors.append(len(samples) / len(X_train))
    print("Class", label, "EM iterations:", len(model.log_likelihoods))
    if len(model.log_likelihoods) == model.max_iter:
        print("Iteration limit reached; convergence is not guaranteed.")


Class 0 EM iterations: 18
Class 1 EM iterations: 17
Class 2 EM iterations: 20
Class 3 EM iterations: 26
Class 4 EM iterations: 27


In [5]:
val_scores = np.column_stack([model.score_samples(X_val) for model in models])
gmm_val_predictions = classes[np.argmax(val_scores + np.log(priors), axis=1)]

test_scores = np.column_stack([model.score_samples(X_test) for model in models])
gmm_test_predictions = classes[np.argmax(test_scores + np.log(priors), axis=1)]


In [6]:
# Confusion matrix
gmm_confusion = np.zeros((len(classes), len(classes)), dtype=int)
for i, actual in enumerate(classes):
    for j, predicted in enumerate(classes):
        gmm_confusion[i, j] = np.sum(
            (y_test == actual) & (gmm_test_predictions == predicted))

print("Validation accuracy:", np.mean(gmm_val_predictions == y_val))
print("Test accuracy:", np.mean(gmm_test_predictions == y_test))
print("Test confusion matrix:\n", gmm_confusion)


Validation accuracy: 0.626
Test accuracy: 0.574
Test confusion matrix:
 [[62  7 16  9  6]
 [ 7 75  8  5  5]
 [ 6  3 68 15  8]
 [ 8  4 29 37 22]
 [ 4  3 18 30 45]]


### **KNN**

In [7]:
knn = KNN(k=5)
knn.fit(X_train, y_train)
knn_val_predictions = knn.predict(X_val)
knn_test_predictions = knn.predict(X_test)


In [8]:
# Confusion matrix
knn_confusion = np.zeros((len(classes), len(classes)), dtype=int)
for i, actual in enumerate(classes):
    for j, predicted in enumerate(classes):
        knn_confusion[i, j] = np.sum(
            (y_test == actual) & (knn_test_predictions == predicted))

print("Validation accuracy:", np.mean(knn_val_predictions == y_val))
print("Test accuracy:", np.mean(knn_test_predictions == y_test))
print("Test confusion matrix:\n", knn_confusion)


Validation accuracy: 0.556
Test accuracy: 0.514
Test confusion matrix:
 [[61 10 18  5  6]
 [ 6 73 14  5  2]
 [ 8  5 67 14  6]
 [11  6 47 26 10]
 [11  6 30 23 30]]
